# Random Message Viewer

Interactive notebook to explore refugee hosting messages randomly.

**Features**:
- Display random messages with formatting
- Show important metadata (location, age, hosting details)
- Filter by quality, region, duration
- Export messages for testing

**Run the data cleaning notebook first**: `02_data_cleaning.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import random
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print("✓ Libraries loaded")

## Load Cleaned Data

In [ ]:
# Load the cleaned dataset
data_path = Path('../data/processed/messages_unified.parquet')

if not data_path.exists():
    # Try CSV
    data_path = Path('../data/processed/messages_unified.csv')
    if not data_path.exists():
        raise FileNotFoundError("Cleaned data not found! Run 02_data_cleaning.ipynb first.")
    df = pd.read_csv(data_path)
else:
    df = pd.read_parquet(data_path)

print(f"✓ Loaded {len(df):,} messages")
print(f"✓ Quality distribution:")
print(f"  High (70-100): {(df['quality_score'] >= 70).sum():,}")
print(f"  Medium (50-69): {((df['quality_score'] >= 50) & (df['quality_score'] < 70)).sum():,}")
print(f"  Low (<50): {(df['quality_score'] < 50).sum():,}")

## Message Display Functions

In [ ]:
# Department to region mapping (simplified)
REGION_MAP = {
    '75': 'Paris', '77': 'Île-de-France', '78': 'Île-de-France', '91': 'Île-de-France',
    '92': 'Île-de-France', '93': 'Île-de-France', '94': 'Île-de-France', '95': 'Île-de-France',
    '13': 'Provence', '06': 'Côte d\'Azur', '83': 'Var', '84': 'Vaucluse',
    '69': 'Lyon/Rhône', '38': 'Grenoble/Isère', '73': 'Savoie', '74': 'Haute-Savoie',
    '59': 'Nord', '62': 'Pas-de-Calais',
    '33': 'Bordeaux/Gironde', '31': 'Toulouse', '34': 'Montpellier',
    '44': 'Nantes', '35': 'Rennes', '29': 'Finistère',
    '67': 'Strasbourg', '68': 'Mulhouse',
}

def get_region(dept):
    """Get region name from department"""
    if pd.isna(dept):
        return 'Unknown'
    return REGION_MAP.get(str(dept), f'Dept. {dept}')

def format_message(row):
    """Format a message for display"""
    
    # Header
    html = f"<div style='border: 2px solid #3498db; border-radius: 10px; padding: 20px; margin: 10px 0; background-color: #f8f9fa;'>"
    html += f"<h3 style='color: #2c3e50; margin-top: 0;'>📨 Message #{row['message_id']} ({row['source_sheet']})</h3>"
    
    # Metadata section
    html += "<div style='background-color: #ecf0f1; padding: 10px; border-radius: 5px; margin-bottom: 15px;'>"
    
    # Location
    if pd.notna(row['department']):
        region = get_region(row['department'])
        html += f"<p><strong>📍 Location:</strong> {region} ({row['department']})" 
        if pd.notna(row['postal_code']):
            html += f" - CP: {row['postal_code']}"
        html += "</p>"
    
    # Age
    if pd.notna(row.get('age')):
        html += f"<p><strong>👤 Age:</strong> {row['age']} ans</p>"
    elif pd.notna(row.get('birth_date')):
        html += f"<p><strong>👤 Birth date:</strong> {row['birth_date']}</p>"
    
    # Space info
    if pd.notna(row['space_category']):
        space_icons = {'room': '🛏️', 'studio': '🏠', 'autonomous': '🏡', 'sofa': '🛋️'}
        icon = space_icons.get(row['space_category'], '📦')
        html += f"<p><strong>{icon} Space:</strong> {row['space_category'].title()}"
        if pd.notna(row.get('space_size_m2')):
            html += f" ({row['space_size_m2']} m²)"
        html += "</p>"
    
    # Duration
    if pd.notna(row['hosting_duration']):
        html += f"<p><strong>⏱️ Duration:</strong> {row['hosting_duration']} ({row['duration_category']})</p>"
    
    # Children
    if pd.notna(row.get('accepts_children')):
        html += f"<p><strong>👶 Accepts children:</strong> {row['accepts_children']}</p>"
    
    # Capacity
    if pd.notna(row.get('hosting_capacity')):
        html += f"<p><strong>👥 Capacity:</strong> {row['hosting_capacity']} person(s)</p>"
    
    html += "</div>"
    
    # Motivation text
    if pd.notna(row['motivation_text']):
        html += "<div style='background-color: #fff3cd; padding: 15px; border-left: 4px solid #ffc107; margin: 15px 0;'>"
        html += "<h4 style='color: #856404; margin-top: 0;'>💭 MOTIVATION</h4>"
        motivation = row['motivation_text'].replace('\n', '<br>')
        html += f"<p style='font-style: italic; color: #333;'>\"{motivation}\"</p>"
        html += "</div>"
    
    # Personal description
    if pd.notna(row['personal_description']):
        html += "<div style='background-color: #d1ecf1; padding: 15px; border-left: 4px solid #17a2b8; margin: 15px 0;'>"
        html += "<h4 style='color: #0c5460; margin-top: 0;'>🙋 PERSONAL DESCRIPTION</h4>"
        description = row['personal_description'].replace('\n', '<br>')
        html += f"<p style='color: #333;'>{description}</p>"
        html += "</div>"
    
    # Other info
    if pd.notna(row.get('other_info')):
        html += "<div style='background-color: #d4edda; padding: 15px; border-left: 4px solid #28a745; margin: 15px 0;'>"
        html += "<h4 style='color: #155724; margin-top: 0;'>ℹ️ OTHER INFORMATION</h4>"
        other = row['other_info'].replace('\n', '<br>')
        html += f"<p style='color: #333;'>{other}</p>"
        html += "</div>"
    
    # Quality footer
    quality_color = '#28a745' if row['quality_score'] >= 70 else '#ffc107' if row['quality_score'] >= 50 else '#dc3545'
    quality_bar = '█' * int(row['quality_score'] / 10) + '░' * (10 - int(row['quality_score'] / 10))
    
    html += "<div style='background-color: #e9ecef; padding: 10px; border-radius: 5px; margin-top: 15px;'>"
    html += f"<p><strong>📊 Quality:</strong> <span style='color: {quality_color}; font-weight: bold;'>{quality_bar} {row['quality_score']}/100</span></p>"
    html += f"<p><strong>📝 Text length:</strong> {row['text_length']} characters</p>"
    html += "</div>"
    
    html += "</div>"
    
    return html

print("✓ Display functions ready")

## Interactive Message Viewer

In [ ]:
# Filtering options
min_quality = widgets.IntSlider(
    value=0,
    min=0,
    max=100,
    step=10,
    description='Min Quality:',
    style={'description_width': 'initial'}
)

duration_filter = widgets.Dropdown(
    options=['all'] + df['duration_category'].unique().tolist(),
    value='all',
    description='Duration:',
    style={'description_width': 'initial'}
)

space_filter = widgets.Dropdown(
    options=['all'] + df['space_category'].unique().tolist(),
    value='all',
    description='Space Type:',
    style={'description_width': 'initial'}
)

region_filter = widgets.Dropdown(
    options=['all'] + sorted([get_region(d) for d in df['department'].unique() if pd.notna(d)]),
    value='all',
    description='Region:',
    style={'description_width': 'initial'}
)

num_messages = widgets.IntSlider(
    value=1,
    min=1,
    max=5,
    step=1,
    description='# Messages:',
    style={'description_width': 'initial'}
)

show_button = widgets.Button(
    description='🎲 Show Random Message(s)',
    button_style='success',
    layout=widgets.Layout(width='300px', height='40px')
)

output = widgets.Output()

def show_random_messages(b=None):
    with output:
        clear_output(wait=True)
        
        # Apply filters
        filtered_df = df[df['quality_score'] >= min_quality.value].copy()
        
        if duration_filter.value != 'all':
            filtered_df = filtered_df[filtered_df['duration_category'] == duration_filter.value]
        
        if space_filter.value != 'all':
            filtered_df = filtered_df[filtered_df['space_category'] == space_filter.value]
        
        if region_filter.value != 'all':
            filtered_df = filtered_df[filtered_df['department'].apply(get_region) == region_filter.value]
        
        if len(filtered_df) == 0:
            print("❌ No messages match the filters!")
            return
        
        print(f"✓ Found {len(filtered_df):,} messages matching filters\n")
        print("="*80)
        
        # Sample random messages
        n = min(num_messages.value, len(filtered_df))
        samples = filtered_df.sample(n=n)
        
        for idx, (_, row) in enumerate(samples.iterrows(), 1):
            if idx > 1:
                print("\n" + "="*80 + "\n")
            display(HTML(format_message(row)))

show_button.on_click(show_random_messages)

# Layout
print("✓ Interactive viewer ready\n")
display(widgets.VBox([
    widgets.HTML("<h2>🎲 Random Message Viewer</h2>"),
    widgets.HTML("<p>Configure filters and click the button to view random messages:</p>"),
    widgets.HBox([min_quality, num_messages]),
    widgets.HBox([duration_filter, space_filter]),
    widgets.HBox([region_filter]),
    show_button,
    output
]))

## Quick Random Samples (No Filters)

In [ ]:
# Show 3 random high-quality messages
print("📋 3 Random High-Quality Messages (Quality >= 70):\n")
print("="*80)

high_quality = df[df['quality_score'] >= 70]
samples = high_quality.sample(n=min(3, len(high_quality)))

for idx, (_, row) in enumerate(samples.iterrows(), 1):
    if idx > 1:
        print("\n" + "="*80 + "\n")
    display(HTML(format_message(row)))

## Export Selected Messages

In [ ]:
# Export high-quality messages for emotion analysis testing
def export_for_testing(n=20, min_quality=70, output_name='test_messages'):
    """
    Export a sample of messages for testing emotion analysis
    """
    test_df = df[
        (df['quality_score'] >= min_quality) & 
        (df['has_motivation'] == True) &
        (df['text_length'] >= 100)
    ].sample(n=min(n, len(df)))
    
    output_path = Path(f'../data/processed/{output_name}.csv')
    test_df.to_csv(output_path, index=False)
    
    print(f"✓ Exported {len(test_df)} messages to: {output_path}")
    print(f"  Quality range: {test_df['quality_score'].min()}-{test_df['quality_score'].max()}")
    print(f"  Text length range: {test_df['text_length'].min()}-{test_df['text_length'].max()} chars")
    
    return test_df

# Example: Export 20 high-quality messages for emotion testing
test_set = export_for_testing(n=20, min_quality=70, output_name='emotion_test_20')

In [ ]:
# Display one from the test set
print("Example message from test set:\n")
display(HTML(format_message(test_set.iloc[0])))

## Statistics by Filter

In [ ]:
# Show distribution of messages by region
print("Top 15 Regions by Message Count:\n")
region_counts = df['department'].apply(get_region).value_counts().head(15)
print(region_counts)

# Visualize
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
region_counts.plot(kind='barh')
plt.xlabel('Number of Messages')
plt.title('Top 15 Regions by Message Count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Text length distribution by quality
print("\nText Length by Quality Category:\n")

df['quality_category'] = pd.cut(df['quality_score'], bins=[0, 50, 70, 100], labels=['Low', 'Medium', 'High'])
print(df.groupby('quality_category')['text_length'].describe())

# Visualize
plt.figure(figsize=(10, 6))
df.boxplot(column='text_length', by='quality_category', figsize=(10, 6))
plt.suptitle('')
plt.title('Text Length Distribution by Quality Category')
plt.ylabel('Text Length (characters)')
plt.xlabel('Quality Category')
plt.tight_layout()
plt.show()

## Next Steps

Now that you can explore the data:

1. ✅ Data is cleaned and unified
2. ✅ Random message viewer working
3. ✅ Export functionality for testing

**Next**: 
- Create `04_emotion_analysis.ipynb` to test emotion detection models
- Build emotional profile extraction
- Map emotions to voice prosody parameters